# Fase 5 - Descomposicion de Series Temporales: Demanda de Taxis

**Pregunta de negocio:** ¿Hay tendencias y estacionalidad en la demanda de taxis?

## Objetivos de este notebook

1. Obtener el conteo diario de viajes para todo 2015 (365 filas)
2. Visualizar la serie temporal cruda y su media/desviacion movil
3. Descomponer la serie en tendencia, estacionalidad y residuos
4. Comparar descomposicion aditiva vs multiplicativa
5. Evaluar estacionariedad con el test ADF (Augmented Dickey-Fuller)
6. Analizar correlogramas ACF y PACF
7. Identificar patrones semanales, efectos de feriados y tendencias estacionales

## ¿Por que analizar series temporales?

Entender los patrones temporales de la demanda es esencial para:
- **Planificacion de flota:** saber cuando se necesitan mas taxis
- **Precios dinamicos:** ajustar tarifas segun la demanda esperada
- **Pronostico:** predecir la demanda futura para tomar mejores decisiones

La descomposicion nos permite separar la serie en componentes interpretables:
tendencia (largo plazo), estacionalidad (patrones repetitivos) y ruido (aleatorio).

## 1. Configuracion e importaciones

In [ ]:
# Conexion a BigQuery
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
# Librerias de analisis y visualizacion
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
warnings.filterwarnings('ignore')

# Configuracion de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Consulta: Conteo diario de viajes para todo 2015

Construimos una serie temporal con el numero de viajes por dia durante
todo el anio 2015. Esto nos dara 365 observaciones.

In [ ]:
query = """
SELECT
    DATE(pickup_datetime) AS trip_date,
    COUNT(*) AS trip_count
FROM
    `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    pickup_datetime >= '2015-01-01'
    AND pickup_datetime < '2016-01-01'
    AND fare_amount > 0
    AND trip_distance > 0
GROUP BY
    trip_date
ORDER BY
    trip_date
"""

df = bq.query_to_df(query)

# Convertir a datetime y establecer como indice
df['trip_date'] = pd.to_datetime(df['trip_date'])
df = df.set_index('trip_date')
df = df.sort_index()

# Asegurar frecuencia diaria (rellenar dias faltantes si los hay)
df = df.asfreq('D', fill_value=0)

print(f"Periodo: {df.index.min().date()} a {df.index.max().date()}")
print(f"Dias totales: {len(df)}")
print(f"\nEstadisticas del conteo diario de viajes:")
print(df['trip_count'].describe().round(0))
df.head(10)

## 3. Visualizacion de la serie temporal

Comenzamos con la visualizacion mas basica: la serie temporal cruda.
Buscamos patrones visibles como tendencias, ciclos y valores atipicos.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.8, alpha=0.8)
ax.fill_between(df.index, df['trip_count'], alpha=0.15, color='#2196F3')

# Marcar feriados importantes de EE.UU.
holidays = {
    '2015-01-01': 'Anio Nuevo',
    '2015-01-26': 'Tormenta Juno',
    '2015-02-14': 'San Valentin',
    '2015-05-25': 'Memorial Day',
    '2015-07-04': 'Independencia',
    '2015-09-07': 'Labor Day',
    '2015-11-26': 'Thanksgiving',
    '2015-12-25': 'Navidad'
}

for date_str, label in holidays.items():
    date = pd.to_datetime(date_str)
    if date in df.index:
        ax.axvline(x=date, color='red', linestyle='--', alpha=0.4)
        ax.annotate(label, xy=(date, df.loc[date, 'trip_count']),
                    xytext=(10, 30), textcoords='offset points',
                    fontsize=8, rotation=45,
                    arrowprops=dict(arrowstyle='->', color='red', alpha=0.6),
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='lightyellow', alpha=0.8))

ax.set_xlabel('Fecha')
ax.set_ylabel('Numero de Viajes')
ax.set_title('Conteo Diario de Viajes de Taxi Amarillo - NYC 2015', fontweight='bold', fontsize=14)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.tight_layout()
plt.show()

print("Observaciones iniciales:")
print("- Se observan caidas pronunciadas en feriados y eventos climaticos")
print("- Existe un patron ciclico semanal visible (oscilaciones regulares)")
print("- La tendencia general parece relativamente estable con variaciones estacionales")

## 4. Media movil y desviacion estandar movil

La **media movil** (rolling mean) de 7 dias suaviza las fluctuaciones semanales
y revela la tendencia subyacente. La **desviacion estandar movil** nos muestra
si la volatilidad cambia con el tiempo.

Si la media movil es constante y la desviacion movil es constante, la serie
podria ser estacionaria.

In [ ]:
# Calcular estadisticas moviles
rolling_mean_7 = df['trip_count'].rolling(window=7).mean()
rolling_std_7 = df['trip_count'].rolling(window=7).std()
rolling_mean_30 = df['trip_count'].rolling(window=30).mean()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Panel superior: serie + medias moviles
ax = axes[0]
ax.plot(df.index, df['trip_count'], color='lightblue', linewidth=0.8, alpha=0.6, label='Diario')
ax.plot(df.index, rolling_mean_7, color='#FF5722', linewidth=2, label='Media movil 7 dias')
ax.plot(df.index, rolling_mean_30, color='#4CAF50', linewidth=2.5, label='Media movil 30 dias')
ax.set_ylabel('Numero de Viajes')
ax.set_title('Serie Temporal con Medias Moviles', fontweight='bold')
ax.legend(loc='upper right')

# Panel inferior: desviacion estandar movil
ax = axes[1]
ax.plot(df.index, rolling_std_7, color='#9C27B0', linewidth=2, label='Desv. estandar movil 7 dias')
ax.fill_between(df.index, rolling_std_7, alpha=0.2, color='#9C27B0')
ax.set_xlabel('Fecha')
ax.set_ylabel('Desviacion Estandar')
ax.set_title('Volatilidad (Desviacion Estandar Movil de 7 Dias)', fontweight='bold')
ax.legend(loc='upper right')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.tight_layout()
plt.show()

print("Observaciones:")
print("- La media movil de 30 dias muestra la tendencia a largo plazo")
print("- La media movil de 7 dias captura variaciones a corto plazo")
print("- Picos de volatilidad coinciden con feriados y eventos climaticos")

## 5. Descomposicion estacional - Modelo Aditivo

La descomposicion aditiva asume que la serie se puede expresar como:

$$Y_t = T_t + S_t + R_t$$

Donde:
- $T_t$ = Tendencia (componente de largo plazo)
- $S_t$ = Estacionalidad (patron repetitivo con periodo fijo)
- $R_t$ = Residuo (ruido aleatorio)

Usamos `period=7` para capturar la estacionalidad semanal (lunes a domingo).

In [ ]:
# Descomposicion aditiva con periodo semanal
decomposition_add = seasonal_decompose(df['trip_count'], model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# Serie original
axes[0].plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.8)
axes[0].set_ylabel('Viajes')
axes[0].set_title('Serie Original', fontweight='bold')

# Tendencia
axes[1].plot(df.index, decomposition_add.trend, color='#FF5722', linewidth=2)
axes[1].set_ylabel('Viajes')
axes[1].set_title('Componente de Tendencia', fontweight='bold')

# Estacionalidad
axes[2].plot(df.index, decomposition_add.seasonal, color='#4CAF50', linewidth=1)
axes[2].set_ylabel('Viajes')
axes[2].set_title('Componente Estacional (Periodo = 7 dias)', fontweight='bold')

# Residuos
axes[3].plot(df.index, decomposition_add.resid, color='#9C27B0', linewidth=0.8, alpha=0.7)
axes[3].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[3].set_ylabel('Viajes')
axes[3].set_title('Componente Residual (Ruido)', fontweight='bold')
axes[3].set_xlabel('Fecha')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.suptitle('Descomposicion Estacional Aditiva (Periodo = 7 dias)',
             fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

print("Interpretacion de la descomposicion aditiva:")
print("- Tendencia: muestra la evolucion a largo plazo de la demanda")
print("- Estacionalidad: patron semanal que se repite (mas viajes entre semana, menos el fin de semana)")
print("- Residuos: variaciones no explicadas por la tendencia ni la estacionalidad")

## 6. Descomposicion estacional - Modelo Multiplicativo

El modelo multiplicativo asume que los componentes se multiplican:

$$Y_t = T_t \times S_t \times R_t$$

Este modelo es apropiado cuando la amplitud de las fluctuaciones estacionales
**crece o decrece proporcionalmente** con el nivel de la serie.

Comparamos ambos modelos para determinar cual se ajusta mejor a nuestros datos.

In [ ]:
# Descomposicion multiplicativa
decomposition_mult = seasonal_decompose(df['trip_count'], model='multiplicative', period=7)

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# Serie original
axes[0].plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.8)
axes[0].set_ylabel('Viajes')
axes[0].set_title('Serie Original', fontweight='bold')

# Tendencia
axes[1].plot(df.index, decomposition_mult.trend, color='#FF5722', linewidth=2)
axes[1].set_ylabel('Viajes')
axes[1].set_title('Componente de Tendencia', fontweight='bold')

# Estacionalidad (multiplicativa: oscila alrededor de 1.0)
axes[2].plot(df.index, decomposition_mult.seasonal, color='#4CAF50', linewidth=1)
axes[2].axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
axes[2].set_ylabel('Factor')
axes[2].set_title('Componente Estacional Multiplicativo (Periodo = 7 dias)', fontweight='bold')

# Residuos
axes[3].plot(df.index, decomposition_mult.resid, color='#9C27B0', linewidth=0.8, alpha=0.7)
axes[3].axhline(y=1.0, color='black', linestyle='--', alpha=0.3)
axes[3].set_ylabel('Factor')
axes[3].set_title('Componente Residual Multiplicativo', fontweight='bold')
axes[3].set_xlabel('Fecha')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.suptitle('Descomposicion Estacional Multiplicativa (Periodo = 7 dias)',
             fontweight='bold', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Comparar residuos de ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

resid_add = decomposition_add.resid.dropna()
resid_mult = decomposition_mult.resid.dropna()

axes[0].hist(resid_add, bins=40, color='#2196F3', alpha=0.7, edgecolor='white')
axes[0].set_title('Residuos - Modelo Aditivo', fontweight='bold')
axes[0].set_xlabel('Residuo')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(x=0, color='red', linestyle='--', alpha=0.7)

axes[1].hist(resid_mult, bins=40, color='#FF5722', alpha=0.7, edgecolor='white')
axes[1].set_title('Residuos - Modelo Multiplicativo', fontweight='bold')
axes[1].set_xlabel('Residuo')
axes[1].set_ylabel('Frecuencia')
axes[1].axvline(x=1, color='red', linestyle='--', alpha=0.7)

plt.suptitle('Comparacion de Residuos: Aditivo vs Multiplicativo',
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Metricas de comparacion
print("Comparacion de modelos (residuos):")
print("=" * 50)
print(f"{'Metrica':<25s} | {'Aditivo':>12s} | {'Multiplicativo':>14s}")
print("-" * 55)
print(f"{'Media de residuos':<25s} | {resid_add.mean():>12.1f} | {resid_mult.mean():>14.4f}")
print(f"{'Desv. est. residuos':<25s} | {resid_add.std():>12.1f} | {resid_mult.std():>14.4f}")
print(f"{'Varianza de residuos':<25s} | {resid_add.var():>12.0f} | {resid_mult.var():>14.4f}")
print(f"\nEl modelo con menor varianza en residuos se ajusta mejor a los datos.")

## 7. Test ADF de estacionariedad

El **test Augmented Dickey-Fuller (ADF)** evalua si una serie temporal es estacionaria.

- **Hipotesis nula (H0):** La serie tiene una raiz unitaria (no es estacionaria)
- **Hipotesis alternativa (H1):** La serie es estacionaria

Si el p-valor < 0.05, rechazamos H0 y concluimos que la serie es estacionaria.

Una serie estacionaria tiene media, varianza y autocovarianza constantes en el tiempo,
lo cual es un requisito para muchos modelos de pronostico (ARIMA, etc.).

In [ ]:
def adf_test(series, title=''):
    """Ejecuta el test ADF y muestra los resultados."""
    result = adfuller(series.dropna(), autolag='AIC')
    
    print(f"Test ADF - {title}")
    print("=" * 50)
    print(f"Estadistico ADF:   {result[0]:.4f}")
    print(f"P-valor:           {result[1]:.6f}")
    print(f"Lags usados:       {result[2]}")
    print(f"Observaciones:     {result[3]}")
    print(f"Valores criticos:")
    for key, value in result[4].items():
        print(f"  {key}: {value:.4f}")
    
    if result[1] < 0.05:
        print(f"\n-> RESULTADO: Serie ESTACIONARIA (p-valor={result[1]:.6f} < 0.05)")
    else:
        print(f"\n-> RESULTADO: Serie NO ESTACIONARIA (p-valor={result[1]:.6f} >= 0.05)")
    print()
    return result

# Test ADF en la serie original
adf_original = adf_test(df['trip_count'], 'Serie Original')

## 8. Primera diferenciacion

Si la serie original no es estacionaria, aplicamos la **primera diferencia**:

$$\Delta Y_t = Y_t - Y_{t-1}$$

Esto elimina la tendencia lineal. Si despues de diferenciar la serie es
estacionaria, decimos que la serie original es **integrada de orden 1** (I(1)),
lo que nos ayuda a definir el parametro d en un modelo ARIMA(p, d, q).

In [ ]:
# Primera diferencia
df['trip_count_diff'] = df['trip_count'].diff()

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Serie original
axes[0].plot(df.index, df['trip_count'], color='#2196F3', linewidth=0.8)
axes[0].set_ylabel('Viajes')
axes[0].set_title('Serie Original', fontweight='bold')

# Serie diferenciada
axes[1].plot(df.index, df['trip_count_diff'], color='#FF5722', linewidth=0.8)
axes[1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
axes[1].set_ylabel('Diferencia de Viajes')
axes[1].set_title('Primera Diferencia (d=1)', fontweight='bold')
axes[1].set_xlabel('Fecha')

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())

plt.suptitle('Serie Original vs Primera Diferencia',
             fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Test ADF en la serie diferenciada
adf_diff = adf_test(df['trip_count_diff'], 'Primera Diferencia')

## 9. ACF y PACF

Los **correlogramas** son herramientas fundamentales en el analisis de series temporales:

- **ACF (Autocorrelation Function):** correlacion de la serie consigo misma a diferentes
  retardos (lags). Incluye correlaciones directas e indirectas.
- **PACF (Partial Autocorrelation Function):** correlacion parcial, eliminando el efecto
  de los retardos intermedios.

Estos graficos nos ayudan a:
- Confirmar la estacionalidad (picos periodicos en el ACF)
- Identificar los parametros p y q de un modelo ARIMA

In [ ]:
# ACF y PACF de la serie original
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ACF - Serie original
plot_acf(df['trip_count'].dropna(), lags=50, ax=axes[0, 0], alpha=0.05,
         title='ACF - Serie Original')
axes[0, 0].axvline(x=7, color='red', linestyle='--', alpha=0.5, label='Lag 7 (semanal)')
axes[0, 0].axvline(x=14, color='red', linestyle='--', alpha=0.3)
axes[0, 0].axvline(x=21, color='red', linestyle='--', alpha=0.3)
axes[0, 0].legend(['Lag 7 (semanal)'], fontsize=9)

# PACF - Serie original
plot_pacf(df['trip_count'].dropna(), lags=50, ax=axes[0, 1], alpha=0.05,
          title='PACF - Serie Original', method='ywm')
axes[0, 1].axvline(x=7, color='red', linestyle='--', alpha=0.5)

# ACF - Primera diferencia
plot_acf(df['trip_count_diff'].dropna(), lags=50, ax=axes[1, 0], alpha=0.05,
         title='ACF - Primera Diferencia')
axes[1, 0].axvline(x=7, color='red', linestyle='--', alpha=0.5)
axes[1, 0].axvline(x=14, color='red', linestyle='--', alpha=0.3)

# PACF - Primera diferencia
plot_pacf(df['trip_count_diff'].dropna(), lags=50, ax=axes[1, 1], alpha=0.05,
          title='PACF - Primera Diferencia', method='ywm')
axes[1, 1].axvline(x=7, color='red', linestyle='--', alpha=0.5)

plt.suptitle('Correlogramas ACF y PACF', fontweight='bold', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print("Interpretacion de los correlogramas:")
print("- Picos significativos en lag 7, 14, 21 confirman estacionalidad SEMANAL")
print("- El ACF decae lentamente en la serie original -> posible no estacionariedad")
print("- El ACF de la primera diferencia decae mas rapido -> mayor estacionariedad")
print("- El PACF ayuda a identificar el orden AR (p) del modelo")

## 10. Analisis de patrones: semanal, feriados, estaciones

Investigamos los patrones identificados en detalle:
1. **Patron semanal:** diferencias entre dias de la semana
2. **Efecto feriados:** caidas de demanda en dias festivos
3. **Tendencia estacional:** verano vs invierno

In [ ]:
# Patron semanal
df['day_of_week'] = df.index.day_of_week
df['day_name'] = df.index.day_name()
df['month'] = df.index.month
df['month_name'] = df.index.month_name()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_spanish = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plot por dia de la semana
weekly_data = df.copy()
weekly_data['day_name'] = pd.Categorical(weekly_data['day_name'], categories=day_order, ordered=True)
sns.boxplot(data=weekly_data, x='day_name', y='trip_count', ax=axes[0],
            palette='Set2', order=day_order)
axes[0].set_xticklabels(day_spanish, rotation=45)
axes[0].set_xlabel('Dia de la Semana')
axes[0].set_ylabel('Viajes Diarios')
axes[0].set_title('Distribucion de Viajes por Dia de la Semana', fontweight='bold')

# Promedio por dia de la semana
day_avg = df.groupby('day_of_week')['trip_count'].mean()
bars = axes[1].bar(range(7), day_avg.values, color='#2196F3', alpha=0.8, edgecolor='white')
axes[1].set_xticks(range(7))
axes[1].set_xticklabels(day_spanish, rotation=45)
axes[1].set_xlabel('Dia de la Semana')
axes[1].set_ylabel('Viajes Promedio')
axes[1].set_title('Promedio de Viajes por Dia de la Semana', fontweight='bold')

# Colorear sabado y domingo diferente
bars[5].set_color('#FF9800')
bars[6].set_color('#FF9800')

# Agregar valores sobre las barras
for bar, val in zip(bars, day_avg.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1000,
                 f'{val:,.0f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("Patron semanal:")
for day_en, day_es in zip(day_order, day_spanish):
    avg = df[df['day_name'] == day_en]['trip_count'].mean()
    print(f"  {day_es:12s}: {avg:,.0f} viajes promedio")

In [ ]:
# Tendencia mensual (verano vs invierno)
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
month_spanish = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

month_avg = df.groupby('month')['trip_count'].agg(['mean', 'std'])

fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#2196F3' if m not in [6, 7, 8] else '#FF5722' for m in range(1, 13)]
bars = ax.bar(range(1, 13), month_avg['mean'], color=colors, alpha=0.8, edgecolor='white',
              yerr=month_avg['std'], capsize=5, error_kw={'alpha': 0.5})

ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_spanish)
ax.set_xlabel('Mes')
ax.set_ylabel('Viajes Diarios Promedio')
ax.set_title('Promedio Diario de Viajes por Mes (Barras naranja = verano)', fontweight='bold')

# Agregar valores
for bar, val in zip(bars, month_avg['mean']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
             f'{val:,.0f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("\nComparacion estacional:")
winter_avg = df[df['month'].isin([12, 1, 2])]['trip_count'].mean()
spring_avg = df[df['month'].isin([3, 4, 5])]['trip_count'].mean()
summer_avg = df[df['month'].isin([6, 7, 8])]['trip_count'].mean()
fall_avg = df[df['month'].isin([9, 10, 11])]['trip_count'].mean()
print(f"  Invierno (Dic-Feb): {winter_avg:,.0f} viajes/dia promedio")
print(f"  Primavera (Mar-May): {spring_avg:,.0f} viajes/dia promedio")
print(f"  Verano (Jun-Ago):   {summer_avg:,.0f} viajes/dia promedio")
print(f"  Otonio (Sep-Nov):   {fall_avg:,.0f} viajes/dia promedio")

In [ ]:
# Efecto de feriados: mostrar los N dias con menor demanda
print("Los 15 dias con menor demanda en 2015:")
print("=" * 60)
lowest_days = df.nsmallest(15, 'trip_count')[['trip_count', 'day_name']]
for date, row in lowest_days.iterrows():
    # Verificar si es un feriado conocido
    date_str = date.strftime('%Y-%m-%d')
    holiday_name = holidays.get(date_str, '')
    marker = f' <- {holiday_name}' if holiday_name else ''
    print(f"  {date.strftime('%Y-%m-%d')} ({row['day_name']:>9s}): {row['trip_count']:>10,} viajes{marker}")

print("\nObservaciones:")
print("- Los dias con menor demanda coinciden con feriados y tormentas de nieve")
print("- La tormenta Juno (enero 2015) causo una caida dramatica en los viajes")
print("- Navidad y Anio Nuevo muestran fuertes caidas de demanda")

## Conclusiones sobre las propiedades de la serie temporal

1. **Estacionalidad semanal confirmada:** Existe un patron claro de 7 dias donde los dias laborales tienen mas viajes que los fines de semana. Los viernes suelen tener el pico de la semana.

2. **Tendencia:** La demanda muestra variaciones mensuales pero sin una tendencia lineal clara de crecimiento o decrecimiento durante 2015.

3. **Efecto feriados:** Los dias festivos y eventos climaticos extremos (como la tormenta Juno de enero) causan caidas significativas en la demanda, apareciendo como outliers en los residuos.

4. **Variacion estacional:** Existe cierta variacion entre meses, con posibles diferencias entre verano e invierno que merecen investigacion adicional.

5. **Estacionariedad:** El test ADF y la diferenciacion nos indican si la serie necesita transformaciones previas al modelado. La primera diferencia suele ser suficiente para lograr estacionariedad.

6. **ACF/PACF:** Los correlogramas confirman la estacionalidad semanal (picos en lags 7, 14, 21) y proporcionan indicaciones para los parametros de un futuro modelo ARIMA o SARIMA.

### Implicaciones practicas
- Planificar mas taxis disponibles entre semana, especialmente los viernes
- Reducir la flota durante feriados y eventos climaticos severos
- Considerar modelos SARIMA con componente estacional de periodo 7 para pronostico

### Proximos pasos
- Fase 6: Modelado predictivo avanzado con series temporales (ARIMA/SARIMA)